In [10]:
import cv2 
import os
import numpy as np
import tensorflow as tf 
from PIL import Image
from sklearn.model_selection import train_test_split
from keras.utils import normalize, to_categorical
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D
from keras.applications import ResNet50
from keras.applications.resnet50 import preprocess_input
from keras.optimizers import Adam
from google.colab import drive, files
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
# 1. Kết nối Drive (Chỉ cần làm 1 lần mỗi buổi)
drive.mount('/content/drive')

# 2. Giải nén dữ liệu (Dùng os.system để tránh lỗi "Expected expression" trong VS Code)
zip_path = "/content/drive/MyDrive/Colab_test/dataset.zip"
if not os.path.exists('/content/dataset'):
    print("Đang giải nén dữ liệu trên server Google...")
    os.system(f'unzip -q "{zip_path}" -d "/content/"')
    print("Giải nén hoàn tất!")

#Cấu hình các tham số
INPUT_SIZE = 224 
image_directory = "/content/dataset/" # Sử dụng đường dẫn trên máy ảo Linux
dataset = []
label = []

no_tumor_images = os.listdir(image_directory + 'no/')
yes_tumor_images = os.listdir(image_directory + 'yes/')

for i, image_name in enumerate(no_tumor_images):
    if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        full_path = os.path.join(image_directory, 'no', image_name)
        image = cv2.imread(full_path)
        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)   
            image = cv2.resize(image, (INPUT_SIZE, INPUT_SIZE))     
            dataset.append(image)
            label.append(0) 

for i, image_name in enumerate(yes_tumor_images):
    if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        full_path = os.path.join(image_directory, 'yes', image_name)
        image = cv2.imread(full_path)
        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (INPUT_SIZE, INPUT_SIZE))
            dataset.append(image)
            label.append(1)

dataset = np.array(dataset)
label = np.array(label)

print(f"\nĐã nạp xong: {len(dataset)} ảnh. Kích thước mảng: {dataset.shape}")

Mounted at /content/drive
Đang giải nén dữ liệu trên server Google...
Giải nén hoàn tất!

Đã nạp xong: 3000 ảnh. Kích thước mảng: (3000, 224, 224, 3)


In [3]:
#chia -> 2 tập + chuẩn hoá
x_train, x_test, y_train, y_test = train_test_split(dataset, label, test_size = 0.2, random_state = 0, shuffle = True)
# gemini gen chuẩn hoá sử dụng preprocess
x_train = preprocess_input(x_train.astype('float32')) 
x_test = preprocess_input(x_test.astype('float32'))

In [ ]:
import gc

# Chỉ xóa nếu biến đã tồn tại trong RAM
if 'dataset' in locals():
    del dataset
if 'label' in locals():
    del label

# Thu gom rác để giải phóng RAM cho Tesla T4
gc.collect()
print("✅ Đã làm sạch bộ nhớ thành công!")

✅ Đã làm sạch bộ nhớ thành công!


In [5]:
#one-hot encode
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

In [6]:
#data_augmentation (off)
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [7]:
#khởi tạo mô hình + tuỳ chỉnh tham số
model = ResNet50(weights='imagenet', include_top=False, input_shape=(INPUT_SIZE, INPUT_SIZE, 3))

# Đóng băng các lớp của mô hình base
for layer in model.layers[:-10]: # để dư 10 layers cuối để mô hình tự cải thiện 
    layer.trainable = False

# Tạo mô hình mới với các lớp tùy chỉnh
model = Sequential([
    model,
    #GlobalAveragePooling2D(), # Sử dụng pooling thay vì flatten 
    Flatten(),
    Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    Dropout(0.5),
    Dense(2, activation='softmax')
])

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [8]:
#khởi chạy
model.compile(
    optimizer = Adam(learning_rate=0.0001), 
    loss = 'categorical_crossentropy', 
    metrics = ['accuracy']
    )
model.fit(
    x_train, y_train, 
    batch_size = 32, 
    epochs = 20, 
    verbose = 1,
    validation_data = (x_test, y_test),
    shuffle = True # loại bỏ học theo thứ tự, trộn 2 tập yes + no random để model up khả năng ứng biến
    )

Epoch 1/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 40s 283ms/step - accuracy: 0.7985 - loss: 10.2156 - val_accuracy: 0.9783 - val_loss: 4.8885
Epoch 2/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 130ms/step - accuracy: 0.9869 - loss: 4.4987 - val_accuracy: 0.9767 - val_loss: 3.6550
Epoch 3/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 131ms/step - accuracy: 0.9975 - loss: 3.3829 - val_accuracy: 0.9867 - val_loss: 2.8666
Epoch 4/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 135ms/step - accuracy: 0.9981 - loss: 2.6776 - val_accuracy: 0.9900 - val_loss: 2.3067
Epoch 5/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 137ms/step - accuracy: 0.9998 - loss: 2.1694 - val_accuracy: 0.9900 - val_loss: 1.8990
Epoch 6/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 139ms/step - accuracy: 0.9989 - loss: 1.7895 - val_accuracy: 0.9867 - val_loss: 1.5900
Epoch 7/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 12s 154ms/step - accuracy: 0.9985 - loss: 1.4933 - val_accuracy: 0.9833 - val_loss: 1.3647
Epoch 8/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 11s 148ms/step - accuracy: 0.9992 - loss: 1.2596 - val_acc

In [ ]:
# Lưu vào bộ nhớ tạm của máy ảo Colab
brain_tumor_result = 'BrainTumor_ResNet50_Final.h5'
model.save(brain_tumor_result)
print("Đã đóng gói xong mô hình!")

# Kích hoạt tải file xuống ổ cứng máy tính
files.download(brain_tumor_result)

Đã đóng gói xong mô hình!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>